In [1]:
import pandas as pd
import numpy as np

# =========================
# 1) Load data
# =========================
df = pd.read_excel("../data/gastat_foreign_trade_cleaned.xlsx")

# =========================
# 2) Helpers (safe)
# =========================
def safe_normalize(s: pd.Series) -> pd.Series:
    """Normalize to 0..1 safely (no NaN explosion if max==min)."""
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() == 0:
        return pd.Series(0.0, index=s.index)
    mn, mx = s.min(), s.max()
    denom = mx - mn
    if pd.isna(denom) or denom == 0:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / denom

def to_5_scale_safe(s: pd.Series) -> pd.Series:
    """Convert numeric series to 1..5 safely (handles duplicates/NaNs/constant series)."""
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

    # If everything is NaN -> neutral
    if s.notna().sum() == 0:
        return pd.Series(3, index=s.index)

    # Fill NaNs with median
    s = s.fillna(s.median())

    # If all values are the same -> neutral
    if s.nunique(dropna=True) == 1:
        return pd.Series(3, index=s.index)

    # Prefer qcut (quantiles) with duplicates handling
    try:
        out = pd.qcut(s, q=5, labels=[1, 2, 3, 4, 5], duplicates="drop")
        # qcut may drop bins and produce NaNs -> fallback
        if out.isna().any():
            raise ValueError("qcut produced NaNs after dropping duplicates")
        return out.astype(int)
    except Exception:
        # Fallback: rank then cut
        r = s.rank(method="average")
        return pd.cut(r, bins=5, labels=[1, 2, 3, 4, 5], include_lowest=True).astype(int)

# =========================
# 3) Build section-level metrics
# =========================
# Total trade per section
section_total = df.groupby("Section")["Million SAR"].sum()

# Share of total trade
total_trade = df["Million SAR"].sum()
section_share = section_total / total_trade if total_trade != 0 else section_total * 0

# Number of countries per section
section_country_count = df.groupby("Section")["Country"].nunique()

# Volatility (std) per section
# ddof=0 reduces NaNs compared to default ddof=1
section_volatility = df.groupby("Section")["Million SAR"].std(ddof=0)

# HHI concentration per section (dependency risk)
hhi_scores = {}
for section, temp in df.groupby("Section"):
    country_sum = temp.groupby("Country")["Million SAR"].sum()
    denom = country_sum.sum()
    if denom == 0 or pd.isna(denom):
        hhi_scores[section] = 0.0
    else:
        share = country_sum / denom
        hhi_scores[section] = float((share ** 2).sum())

section_hhi = pd.Series(hhi_scores)

# =========================
# 4) Normalize metrics safely
# =========================
norm_total = safe_normalize(section_total)
norm_share = safe_normalize(section_share)
norm_country_count = safe_normalize(section_country_count)
norm_volatility = safe_normalize(section_volatility)
norm_hhi = safe_normalize(section_hhi)

# =========================
# 5) Compute scores (0..1-ish)
# =========================
# Criticality: importance + size + dependency
criticality = (0.5 * norm_share) + (0.3 * norm_total) + (0.2 * norm_hhi)

# Complexity: more countries + more volatility + more share
complexity = (0.4 * norm_country_count) + (0.4 * norm_volatility) + (0.2 * norm_share)

# Ease: inverse of volatility and dependency (higher is easier)
ease = ((1 - norm_volatility) + (1 - norm_hhi)) / 2

# =========================
# 6) Scale to 1..5
# =========================
rules = pd.DataFrame({
    "Section": criticality.index,
    "Criticality": to_5_scale_safe(criticality),
    "Complexity": to_5_scale_safe(complexity),
    "Ease": to_5_scale_safe(ease),
})

# =========================
# 7) Add Section ID (from your file mapping) + reorder
# =========================
section_map = (
    df[["Section ID", "Section"]]
    .drop_duplicates()
    .set_index("Section")["Section ID"]
)

rules["Section ID"] = rules["Section"].map(section_map)
rules = rules[["Section ID", "Section", "Criticality", "Complexity", "Ease"]].sort_values(
    ["Criticality", "Complexity"], ascending=[False, False]
).reset_index(drop=True)

# =========================
# 8) Save outputs
# =========================
rules.to_csv("Criticality,Complexity,Ease_Table", index=False, encoding="utf-8-sig")
rules.to_excel("Criticality,Complexity,Ease_Table.xlsx", index=False)

# =========================
# 9) Show result
# =========================
print("✅ Done. Saved: rules.csv + rules.xlsx")
display(rules)

✅ Done. Saved: rules.csv + rules.xlsx


,Section ID,Section,Criticality,Complexity,Ease
0,S16,"Machinery and mechanical appliances, electrica...",5,5,1
1,S07,"Plastics, rubber and their articles",5,5,3
2,S06,Products of the chemical industries,5,5,3
3,S17,"Vehicles, aircraft, vessels and associated tra...",5,5,2
4,S15,Base metals and their articles,4,4,3
5,S05,Mineral Products,4,4,3
6,S14,"Precious stones or metals and their articles, ...",4,4,1
7,S12,"Footwear, headgear, umbrellas, sticks",4,1,1
8,S11,Textiles and their articles,3,4,2
9,S20,Miscellaneous Manufactured Articles,3,3,2
